In [3]:
# Cell 1: Install dependencies (takes ~2-3 min on first run)
!pip install -q numpy torch owlready2 sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 23.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729.6/72

In [5]:
# Cell 2: Verify GPU availability (optional)
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU — mock LLM will be used.")

CUDA available: False
Running on CPU — mock LLM will be used.


In [8]:
"""
Enhanced Neuro-Symbolic Framework for LLM-Ontology Integration
================================================================
Hybrid Asynchronous Verification Architecture
(Colab-ready, self-contained, no CLI arguments)
"""

from __future__ import annotations

import asyncio
import logging
import re
import sys
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple, Union

# ------------------------------------------------------------------
# Force UTF-8 output (prevents UnicodeEncodeError in some environments)
# ------------------------------------------------------------------
try:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

# ------------------------------------------------------------------
# Required dependency
# ------------------------------------------------------------------
try:
    import numpy as np
except ImportError as exc:
    raise SystemExit("numpy is required:  !pip install numpy") from exc

# ------------------------------------------------------------------
# Optional dependencies with graceful fallback
# ------------------------------------------------------------------
try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    class _DummyModule:
        def __init__(self, *a, **k): pass
    class _DummyNN:
        Module = _DummyModule
    nn = _DummyNN()
    torch = None

try:
    import owlready2
    OWLREADY2_AVAILABLE = True
except ImportError:
    OWLREADY2_AVAILABLE = False

try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False

try:
    from sentence_transformers import SentenceTransformer
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    SENTENCE_TRANSFORMERS_AVAILABLE = False

# ------------------------------------------------------------------
# Logging
# ------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("neurosym")


# ==================================================================
# Enums & Data Classes
# ==================================================================

class DomainType(Enum):
    BIOMEDICAL = "biomedical"
    FINANCIAL  = "financial"
    LEGAL      = "legal"
    GENERAL    = "general"


class VerificationPath(Enum):
    FAST        = "fast"
    CORRECTION  = "correction"
    SLOW        = "slow"
    HUMAN_REVIEW = "human_review"


@dataclass
class ThresholdConfig:
    theta_low_base: float = 0.65
    theta_epistemic_base: float = 0.45
    alpha: float = 0.05
    beta: float = 0.10
    gamma: float = 0.04
    delta: float = 0.08
    ref_ontology_size: int = 5000

    theta_low_min: float = 0.3
    theta_low_max: float = 0.9
    theta_epistemic_min: float = 0.1
    theta_epistemic_max: float = 0.8

    @classmethod
    def for_domain(cls, domain: "DomainType") -> "ThresholdConfig":
        presets = {
            DomainType.BIOMEDICAL: dict(
                theta_low_base=0.65, theta_epistemic_base=0.45,
                alpha=0.05, beta=0.10, gamma=0.04, delta=0.08,
                ref_ontology_size=5000),
            DomainType.FINANCIAL: dict(
                theta_low_base=0.55, theta_epistemic_base=0.40,
                alpha=0.04, beta=0.08, gamma=0.03, delta=0.06,
                ref_ontology_size=2800),
            DomainType.LEGAL: dict(
                theta_low_base=0.75, theta_epistemic_base=0.55,
                alpha=0.06, beta=0.12, gamma=0.05, delta=0.10,
                ref_ontology_size=10000),
            DomainType.GENERAL: dict(),
        }
        return cls(**presets.get(domain, {}))


@dataclass
class Triple:
    subject: str
    predicate: str
    object: str
    confidence: float = 1.0
    source: str = "llm"

    def to_tuple(self) -> Tuple[str, str, str]:
        return (self.subject, self.predicate, self.object)

    def to_text(self) -> str:
        return f"{self.subject} {self.predicate} {self.object}"

    def __hash__(self) -> int:
        return hash((self.subject, self.predicate, self.object))


@dataclass
class QueryResult:
    query_id: str
    query: str
    response: str
    fast_score: float
    theta_low: float
    theta_epistemic: float
    path: VerificationPath
    latency_ms: float
    triples: List[Triple] = field(default_factory=list)
    corrections: List[str] = field(default_factory=list)
    epistemic_trigger: float = 0.0
    metadata: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "query_id": self.query_id,
            "query": self.query,
            "response": self.response,
            "fast_score": self.fast_score,
            "theta_low": self.theta_low,
            "theta_epistemic": self.theta_epistemic,
            "path": self.path.value,
            "latency_ms": self.latency_ms,
            "triples": [t.to_tuple() for t in self.triples],
            "corrections": self.corrections,
            "epistemic_trigger": self.epistemic_trigger,
            "metadata": self.metadata,
        }


@dataclass
class SystemStatistics:
    fast_path_count: int = 0
    slow_path_count: int = 0
    correction_count: int = 0
    evolution_triggers: int = 0
    total_queries: int = 0

    fast_path_latencies: List[float] = field(default_factory=list)
    slow_path_latencies: List[float] = field(default_factory=list)
    correction_latencies: List[float] = field(default_factory=list)

    true_positives: int = 0
    false_positives: int = 0
    true_negatives: int = 0
    false_negatives: int = 0

    def record_fast_path(self, latency_ms: float) -> None:
        self.fast_path_count += 1
        self.fast_path_latencies.append(latency_ms)
        self.total_queries += 1

    def record_slow_path(self, latency_ms: float) -> None:
        self.slow_path_count += 1
        self.slow_path_latencies.append(latency_ms)

    def record_correction(self, latency_ms: float) -> None:
        self.correction_count += 1
        self.correction_latencies.append(latency_ms)
        self.total_queries += 1

    def record_evolution_trigger(self) -> None:
        self.evolution_triggers += 1

    def get_summary(self) -> Dict[str, Any]:
        def safe_stats(vals: List[float]) -> Dict[str, float]:
            if not vals:
                return {"mean": 0.0, "std": 0.0, "min": 0.0,
                        "max": 0.0, "count": 0}
            return {
                "mean": float(np.mean(vals)),
                "std":  float(np.std(vals)),
                "min":  float(np.min(vals)),
                "max":  float(np.max(vals)),
                "count": len(vals),
            }
        total = self.fast_path_count + self.correction_count
        return {
            "total_queries": self.total_queries,
            "fast_path_count": self.fast_path_count,
            "slow_path_count": self.slow_path_count,
            "correction_count": self.correction_count,
            "evolution_triggers": self.evolution_triggers,
            "fast_path_ratio": self.fast_path_count / total if total else 0.0,
            "correction_rate": self.correction_count / total if total else 0.0,
            "evolution_rate": self.evolution_triggers / max(total, 1),
            "fast_path_latency":  safe_stats(self.fast_path_latencies),
            "slow_path_latency":  safe_stats(self.slow_path_latencies),
            "correction_latency": safe_stats(self.correction_latencies),
            "accuracy_metrics": {
                "precision": self.true_positives / max(
                    self.true_positives + self.false_positives, 1),
                "recall": self.true_positives / max(
                    self.true_positives + self.false_negatives, 1),
                "f1": 2 * self.true_positives / max(
                    2 * self.true_positives
                    + self.false_positives + self.false_negatives, 1),
            },
        }


# ==================================================================
# Core Mathematical Functions
# ==================================================================

class SemanticConsistencyFunction:
    """S(T,O) = 1 - |{(s,r,o) : O ∪ {(s,r,o)} ⊢ ⊥}| / |T|"""

    def __init__(self, ontology: Optional[Any] = None):
        self.ontology = ontology
        self._reasoner_available = OWLREADY2_AVAILABLE and ontology is not None

    def compute(self, triples: List[Triple],
                ontology: Optional[Any] = None) -> float:
        if not triples:
            return 1.0
        target = ontology if ontology is not None else self.ontology
        if target is None:
            return self._approximate_consistency(triples)
        contradictions = sum(
            1 for t in triples if self._check_contradiction(t, target))
        return 1.0 - (contradictions / len(triples))

    def _check_contradiction(self, triple: Triple, ontology: Any) -> bool:
        return False  # real DL reasoner would be invoked here

    def _approximate_consistency(self, triples: List[Triple]) -> float:
        return 0.85


class EpistemicDiscoveryTrigger:
    """Δ_epistemic = H(P_θ) × (1 - S(τ(Y), O))"""

    def __init__(self, consistency_fn: Optional[SemanticConsistencyFunction] = None):
        self.consistency_fn = consistency_fn or SemanticConsistencyFunction()

    def compute(self, entropy: float, triples: List[Triple],
                ontology: Optional[Any] = None) -> float:
        consistency = self.consistency_fn.compute(triples, ontology)
        return entropy * (1.0 - consistency)

    @staticmethod
    def compute_entropy(logits: np.ndarray, temperature: float = 1.0) -> float:
        scaled = logits / temperature
        exp_logits = np.exp(scaled - np.max(scaled))
        probs = exp_logits / np.sum(exp_logits)
        return float(-np.sum(probs * np.log(probs + 1e-10)))


class AxiomWeakeningOperator:
    """W_α : A → A"""

    def __init__(self) -> None:
        self.weakening_strategies = {
            "subclass":    self._weaken_subclass,
            "existential": self._weaken_existential,
            "universal":   self._weaken_universal,
            "cardinality": self._weaken_cardinality,
        }

    def weaken(self, axiom: Any, strategy: str = "subclass") -> Any:
        fn = self.weakening_strategies.get(strategy)
        return fn(axiom) if fn else axiom

    def _weaken_subclass(self, axiom):    return axiom
    def _weaken_existential(self, axiom): return axiom
    def _weaken_universal(self, axiom):   return axiom
    def _weaken_cardinality(self, axiom): return axiom


# ==================================================================
# Differentiable Reasoning (only meaningful if torch is available)
# ==================================================================

if TORCH_AVAILABLE:

    class NeuroSymbolicLoss(nn.Module):
        def __init__(self, lambda_symbolic: float = 0.1):
            super().__init__()
            self.lambda_symbolic = lambda_symbolic
            self.nll_loss = nn.NLLLoss()
            self.consistency_fn = SemanticConsistencyFunction()

        def forward(self, log_probs, targets, triples):
            batch, seq, vocab = log_probs.shape
            nll = self.nll_loss(
                log_probs.view(-1, vocab), targets.view(-1))
            consistency = self.consistency_fn.compute(triples)
            penalty = 1.0 - consistency
            total = nll + self.lambda_symbolic * penalty
            return total, {
                "nll": float(nll.item()),
                "consistency": consistency,
                "consistency_penalty": penalty,
                "total": float(total.item()),
            }

    class StraightThroughEstimator(nn.Module):
        def __init__(self, temperature: float = 1.0):
            super().__init__()
            self.temperature = temperature

        def forward(self, logits):
            hard = (logits > 0).float()
            soft = torch.sigmoid(logits / self.temperature)
            return hard + (soft - soft.detach())

        def estimate_gradient(self, logits, consistency_score):
            epsilon = 1e-4
            return (consistency_score
                    - consistency_score.detach()) / epsilon

    class DifferentiableReasoner(nn.Module):
        def __init__(self, num_predicates=100, embedding_dim=256, temperature=1.0):
            super().__init__()
            self.embedding_dim = embedding_dim
            self.predicate_embeddings = nn.Embedding(num_predicates, embedding_dim)
            self.and_gate = nn.Sequential(
                nn.Linear(embedding_dim * 2, embedding_dim), nn.ReLU(),
                nn.Linear(embedding_dim, 1), nn.Sigmoid())
            self.or_gate = nn.Sequential(
                nn.Linear(embedding_dim * 2, embedding_dim), nn.ReLU(),
                nn.Linear(embedding_dim, 1), nn.Sigmoid())
            self.not_gate = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim), nn.ReLU(),
                nn.Linear(embedding_dim, 1), nn.Sigmoid())
            self.ste = StraightThroughEstimator(temperature)

        def forward(self, predicate_ids, logical_operations):
            result = self.predicate_embeddings(predicate_ids)
            for op in logical_operations:
                if op == "and":   result = self._apply_and(result)
                elif op == "or":  result = self._apply_or(result)
                elif op == "not": result = self._apply_not(result)
            return result

        def _apply_and(self, e):
            if e.shape[1] < 2: return e
            pairs = torch.stack([
                e[:, i:i+2, :].reshape(-1, self.embedding_dim*2)
                for i in range(0, e.shape[1]-1, 2)], dim=1)
            return self.and_gate(pairs)

        def _apply_or(self, e):
            if e.shape[1] < 2: return e
            pairs = torch.stack([
                e[:, i:i+2, :].reshape(-1, self.embedding_dim*2)
                for i in range(0, e.shape[1]-1, 2)], dim=1)
            return self.or_gate(pairs)

        def _apply_not(self, e):
            return self.not_gate(e)

else:
    class NeuroSymbolicLoss:      # noqa
        def __init__(self, *a, **k): pass
    class StraightThroughEstimator:  # noqa
        def __init__(self, *a, **k): pass
    class DifferentiableReasoner:  # noqa
        def __init__(self, *a, **k): pass


# ==================================================================
# Ontology Manager
# ==================================================================

class OntologyManager:
    def __init__(self, ontology_path: Optional[str] = None):
        self.ontology = None
        self.ontology_path = ontology_path
        self.classes: List[str] = []
        self.properties: List[str] = []
        self.axioms: List[str] = []
        self._reasoner_available = False

        if ontology_path and OWLREADY2_AVAILABLE:
            self.load(ontology_path)
        else:
            if ontology_path and not OWLREADY2_AVAILABLE:
                logger.warning("owlready2 not installed; using mock ontology.")
            self._create_mock_ontology()

    def load(self, path: str) -> bool:
        try:
            self.ontology = owlready2.get_ontology(path).load()
            self.classes = [str(c) for c in self.ontology.classes()]
            self.properties = [str(p) for p in self.ontology.properties()]
            self.axioms = [str(a) for a in self.ontology.axioms()]
            self._reasoner_available = True
            logger.info("Loaded ontology: %d classes, %d properties, %d axioms",
                        len(self.classes), len(self.properties), len(self.axioms))
            return True
        except Exception as exc:
            logger.error("Failed to load %s: %s", path, exc)
            self._create_mock_ontology()
            return False

    def _create_mock_ontology(self) -> None:
        self.classes = [
            "Disease", "Symptom", "Treatment", "Patient", "Diagnosis",
            "Medication", "Procedure", "Anatomy", "Physiology", "Pathology",
        ]
        self.properties = [
            "has_symptom", "treated_by", "diagnosed_with", "causes",
            "affects", "contraindicated_with", "indicated_for",
        ]
        self.axioms = [
            "Disease subClassOf has_symptom some Symptom",
            "Treatment subClassOf treated_by some Disease",
            "Patient subClassOf diagnosed_with some Disease",
        ]
        self._reasoner_available = False

    def check_consistency(self, triples: List[Triple]) -> Tuple[bool, List[str]]:
        if not self._reasoner_available:
            return True, []
        return True, []

    def get_complexity_score(self) -> float:
        ex = sum(1 for ax in self.axioms if "some" in ax.lower())
        un = sum(1 for ax in self.axioms if "only" in ax.lower())
        return ex / (un + 1)

    def get_size(self) -> int:
        return len(self.classes)

    def add_class(self, name: str) -> bool:
        if name not in self.classes:
            self.classes.append(name); return True
        return False

    def add_axiom(self, axiom: str) -> bool:
        if axiom not in self.axioms:
            self.axioms.append(axiom); return True
        return False


# ==================================================================
# Triple Extraction
# ==================================================================

class TripleExtractor:
    def __init__(self, strategy: str = "pattern"):
        self.strategy = strategy
        self._patterns = [
            (r"(\w+)\s+(?:is|are)\s+(?:a|an)\s+(\w+)", "is_a"),
            (r"(\w+)\s+(?:has|have|had)\s+(\w+)",       "has"),
            (r"(\w+)\s+(?:causes?|caused)\s+(\w+)",    "causes"),
            (r"(\w+)\s+(?:treats?|treated)\s+(\w+)",   "treats"),
            (r"(\w+)\s+leads?\s+to\s+(\w+)",           "leads_to"),
        ]

    def extract(self, text: str) -> List[Triple]:
        if not text or not text.strip():
            return []
        if self.strategy == "pattern":
            return self._pattern_extract(text)
        return self._simple_extract(text)

    def _pattern_extract(self, text: str) -> List[Triple]:
        triples: List[Triple] = []
        for pat, pred in self._patterns:
            for m in re.finditer(pat, text, re.IGNORECASE):
                g = m.groups()
                if len(g) >= 2 and g[0] and g[1]:
                    triples.append(Triple(
                        subject=g[0], predicate=pred, object=g[1],
                        confidence=0.7, source="pattern"))
        return triples if triples else self._simple_extract(text)

    def _simple_extract(self, text: str) -> List[Triple]:
        triples: List[Triple] = []
        for sent in re.split(r"[.!?]+", text):
            words = [w for w in sent.strip().split() if w]
            if len(words) >= 3:
                triples.append(Triple(
                    subject=words[0], predicate="related_to",
                    object=words[-1], confidence=0.5, source="simple"))
        return triples


# ==================================================================
# Embedding Manager
# ==================================================================

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.embedding_dim = 384

        if SENTENCE_TRANSFORMERS_AVAILABLE:
            try:
                self.model = SentenceTransformer(model_name)
                self.embedding_dim = self.model.get_sentence_embedding_dimension()
                logger.info("Loaded embedding model: %s (dim=%d)",
                            model_name, self.embedding_dim)
            except Exception as exc:
                logger.warning("Embedding model load failed: %s. Using mock.", exc)
                self.model = None

    def encode(self, texts: Union[str, List[str]]) -> np.ndarray:
        if isinstance(texts, str):
            texts = [texts]
        if self.model is not None:
            try:
                return self.model.encode(texts, convert_to_numpy=True)
            except Exception as exc:
                logger.warning("Embedding failed (%s); using mock.", exc)
        return self._random_encode(texts)

    def _random_encode(self, texts: List[str]) -> np.ndarray:
        out = []
        for t in texts:
            seed = abs(hash(t)) % (2**32)
            rng = np.random.RandomState(seed)
            v = rng.randn(self.embedding_dim).astype(np.float32)
            v /= (np.linalg.norm(v) + 1e-10)
            out.append(v)
        return np.array(out)

    @staticmethod
    def compute_similarity(q: np.ndarray, t: np.ndarray) -> np.ndarray:
        q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-10)
        t = t / (np.linalg.norm(t, axis=1, keepdims=True) + 1e-10)
        return np.dot(q, t.T)


# ==================================================================
# Main Neuro-Symbolic System
# ==================================================================

class NeuroSymbolicSystem:
    """
    Hybrid Asynchronous Verification Architecture for LLM-Ontology Integration.
    """

    def __init__(
        self,
        ontology_path: Optional[str] = None,
        llm_model_name: str = "meta-llama/Llama-3-8B",
        embedding_model_name: str = "all-MiniLM-L6-v2",
        threshold_config: Optional[ThresholdConfig] = None,
        domain: DomainType = DomainType.GENERAL,
        enable_async: bool = True,
    ):
        self.system_id = str(uuid.uuid4())[:8]
        self.domain = domain
        self.enable_async = enable_async

        self.ontology_manager = OntologyManager(ontology_path)
        self.triple_extractor = TripleExtractor()
        self.embedding_manager = EmbeddingManager(embedding_model_name)
        self.consistency_fn = SemanticConsistencyFunction(
            self.ontology_manager.ontology)
        self.epistemic_trigger = EpistemicDiscoveryTrigger(self.consistency_fn)
        self.axiom_weakening = AxiomWeakeningOperator()

        self.threshold_config = (
            threshold_config or ThresholdConfig.for_domain(domain))

        self.llm = None
        self.sampling_params = None
        self._initialize_llm(llm_model_name)

        self.ontology_embeddings = self._compute_ontology_embeddings()

        self.reasoner_queue: "asyncio.Queue[Any]" = asyncio.Queue()
        self.human_review_queue: "asyncio.Queue[Any]" = asyncio.Queue()
        self.evolution_queue: "asyncio.Queue[Any]" = asyncio.Queue()

        self.stats = SystemStatistics()

        logger.info("NeuroSymbolicSystem [%s] initialised (domain=%s, async=%s)",
                    self.system_id, domain.value, enable_async)

    # ------------------------------------------------------------------
    def _initialize_llm(self, model_name: str) -> None:
        if VLLM_AVAILABLE:
            try:
                self.llm = LLM(model=model_name)
                self.sampling_params = SamplingParams(
                    temperature=0.7, max_tokens=512, top_p=0.9)
                logger.info("Initialised LLM: %s", model_name)
            except Exception as exc:
                logger.warning("Failed to init LLM: %s", exc)
                self.llm = None
        else:
            logger.info("vLLM not available; using mock LLM responses.")

    def _compute_ontology_embeddings(self) -> np.ndarray:
        concepts = self.ontology_manager.classes or \
                   ["Disease", "Symptom", "Treatment"]
        self.ontology_manager.classes = concepts
        embeddings = self.embedding_manager.encode(concepts)
        logger.info("Computed embeddings for %d concepts", len(concepts))
        return embeddings

    def _compute_dynamic_thresholds(self) -> Tuple[float, float]:
        size = max(self.ontology_manager.get_size(), 1)
        kappa = self.ontology_manager.get_complexity_score()
        cfg = self.threshold_config
        size_ratio = np.log(size / cfg.ref_ontology_size) if size > 0 else 0.0

        theta_low = cfg.theta_low_base * (
            1 + cfg.alpha * size_ratio + cfg.beta * kappa)
        theta_epi = cfg.theta_epistemic_base * (
            1 + cfg.gamma * size_ratio + cfg.delta * kappa)

        theta_low = float(np.clip(theta_low,
                                  cfg.theta_low_min, cfg.theta_low_max))
        theta_epi = float(np.clip(theta_epi,
                                  cfg.theta_epistemic_min,
                                  cfg.theta_epistemic_max))
        return theta_low, theta_epi

    def _generate_response(self, query: str) -> str:
        if self.llm is not None and VLLM_AVAILABLE:
            try:
                out = self.llm.generate([query], self.sampling_params)
                return out[0].outputs[0].text
            except Exception as exc:
                logger.error("LLM generation failed: %s", exc)
        return self._mock_response(query)

    @staticmethod
    def _mock_response(query: str) -> str:
        responses = {
            "diabetes":  ("Diabetes is a chronic condition characterized by "
                          "elevated blood sugar. It can lead to cardiovascular "
                          "disease. Treatment includes insulin therapy."),
            "symptom":   ("Common symptoms include fatigue, increased thirst, "
                          "and frequent urination. These symptoms may indicate "
                          "underlying conditions."),
            "treatment": ("Treatment options include medication, therapy, and "
                          "lifestyle changes. Consultation with a healthcare "
                          "provider is recommended."),
            "asthma":    ("Asthma is a chronic respiratory disease that causes "
                          "airway inflammation. It is treated with bronchodilators."),
            "hypertension": ("Hypertension is diagnosed through repeated blood "
                             "pressure measurements. It causes cardiovascular disease."),
        }
        q = query.lower()
        for key, resp in responses.items():
            if key in q:
                return resp
        return (f"Based on the query about {query[:50]}, the relevant "
                f"information suggests a relationship between the concepts "
                f"involved. Further analysis may be required.")

    def _vector_similarity_check(self, triples: List[Triple]) -> float:
        if not triples:
            return 1.0
        emb = self.embedding_manager.encode([t.to_text() for t in triples])
        sims = self.embedding_manager.compute_similarity(
            emb, self.ontology_embeddings)
        return float(np.clip(np.mean(np.max(sims, axis=1)), 0.0, 1.0))

    def _generate_correction(
        self, original: str, triples: List[Triple],
        contradictions: Optional[List[str]] = None,
    ) -> str:
        msg = ("I need to verify some information. "
               f"The following response may contain inconsistencies: "
               f"'{original[:200]}...'")
        if contradictions:
            msg += f" Issues: {'; '.join(contradictions[:3])}."
        msg += " Please provide a more accurate response based on the ontology."
        return msg

    async def _async_verification(
        self, query_id: str, llm_output: str,
        triples: List[Triple], theta_epistemic: float,
    ) -> None:
        start = time.time()
        try:
            ok, contradictions = self.ontology_manager.check_consistency(triples)
            if not ok:
                entropy = 0.5
                delta = self.epistemic_trigger.compute(entropy, triples)
                if delta > theta_epistemic:
                    await self._trigger_evolution(query_id, contradictions, triples)
                    self.stats.record_evolution_trigger()
            self.stats.record_slow_path((time.time() - start) * 1000)
        except Exception as exc:
            logger.error("Async verification failed for %s: %s", query_id, exc)

    async def _trigger_evolution(
        self, query_id: str, contradictions: List[str], triples: List[Triple],
    ) -> None:
        weakened = [self.axiom_weakening.weaken(c) for c in contradictions]
        await self.human_review_queue.put({
            "query_id": query_id,
            "contradictions": contradictions,
            "weakened_axioms": weakened,
            "triples": [t.to_tuple() for t in triples],
            "timestamp": datetime.now().isoformat(),
            "status": "pending",
        })
        logger.info("Queued ontology evolution for %s", query_id)

    def process_query(self, query: str, run_async: bool = True) -> QueryResult:
        start = time.time()
        qid = str(uuid.uuid4())[:8]

        theta_low, theta_epi = self._compute_dynamic_thresholds()
        llm_out = self._generate_response(query)
        triples = self.triple_extractor.extract(llm_out)
        fast_score = self._vector_similarity_check(triples)

        if fast_score < theta_low:
            path = VerificationPath.CORRECTION
            response = self._generate_correction(llm_out, triples)
            self.stats.record_correction((time.time() - start) * 1000)
        else:
            path = VerificationPath.FAST
            response = llm_out
            self.stats.record_fast_path((time.time() - start) * 1000)

        result = QueryResult(
            query_id=qid, query=query, response=response,
            fast_score=fast_score, theta_low=theta_low,
            theta_epistemic=theta_epi, path=path,
            latency_ms=(time.time() - start) * 1000,
            triples=triples,
            metadata={
                "domain": self.domain.value,
                "ontology_size": self.ontology_manager.get_size(),
                "complexity_score": self.ontology_manager.get_complexity_score(),
            },
        )

        if run_async and self.enable_async and path == VerificationPath.FAST:
            self._schedule_async(result, llm_out, theta_epi)

        return result

    def _schedule_async(self, result: QueryResult,
                        llm_output: str, theta_epi: float) -> None:
        coro = self._async_verification(
            result.query_id, llm_output, result.triples, theta_epi)
        try:
            loop = asyncio.get_running_loop()
            loop.create_task(coro)
        except RuntimeError:
            try:
                asyncio.run(coro)
            except Exception as exc:
                logger.debug("Sync fallback for async verification: %s", exc)

    async def process_query_async(self, query: str) -> QueryResult:
        return self.process_query(query, run_async=True)

    def get_statistics(self) -> Dict[str, Any]:
        return self.stats.get_summary()

    def reset_statistics(self) -> None:
        self.stats = SystemStatistics()

    async def get_human_review_items(self) -> List[Dict[str, Any]]:
        items: List[Dict[str, Any]] = []
        while not self.human_review_queue.empty():
            items.append(await self.human_review_queue.get())
        return items


# ==================================================================
# Benchmarking Suite
# ==================================================================

class BenchmarkSuite:
    def __init__(self, system: NeuroSymbolicSystem):
        self.system = system
        self.results: Dict[str, Any] = {}

    def run_latency_benchmark(self, queries: List[str],
                              num_runs: int = 50) -> Dict[str, Any]:
        fast, corr = [], []
        for q in queries:
            for _ in range(num_runs):
                r = self.system.process_query(q, run_async=False)
                if r.path == VerificationPath.FAST:
                    fast.append(r.latency_ms)
                elif r.path == VerificationPath.CORRECTION:
                    corr.append(r.latency_ms)

        total_s = sum(fast) / 1000 if fast else 0
        throughput = len(fast) / total_s if total_s else 0

        out = {
            "fast_path": {
                "mean_ms": float(np.mean(fast)) if fast else 0.0,
                "std_ms":  float(np.std(fast)) if fast else 0.0,
                "count":   len(fast),
            },
            "correction_path": {
                "mean_ms": float(np.mean(corr)) if corr else 0.0,
                "std_ms":  float(np.std(corr)) if corr else 0.0,
                "count":   len(corr),
            },
            "throughput_qps": throughput,
        }
        self.results["latency"] = out
        return out

    def run_accuracy_benchmark(self, queries: List[str],
                               ground_truth: Dict[str, bool]) -> Dict[str, Any]:
        correct = total = 0
        for q in queries:
            r = self.system.process_query(q, run_async=False)
            expected = ground_truth.get(q, True)
            predicted = r.fast_score >= r.theta_low
            correct += int(expected == predicted)
            total += 1
        out = {
            "accuracy": correct / total if total else 0.0,
            "correct": correct, "total": total,
        }
        self.results["accuracy"] = out
        return out

    def run_threshold_calibration(self, queries: List[str],
                                  true_labels: List[bool]) -> Dict[str, Any]:
        scores = [self.system.process_query(q, run_async=False).fast_score
                  for q in queries]
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.3, 0.9, 0.05):
            preds = [s >= t for s in scores]
            tp = sum(1 for p, x in zip(preds, true_labels) if p and x)
            fp = sum(1 for p, x in zip(preds, true_labels) if p and not x)
            fn = sum(1 for p, x in zip(preds, true_labels) if not p and x)
            prec = tp / (tp + fp) if (tp + fp) else 0
            rec  = tp / (tp + fn) if (tp + fn) else 0
            f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
            if f1 > best_f1:
                best_f1, best_t = f1, t
        out = {
            "optimal_threshold": float(best_t),
            "best_f1": float(best_f1),
            "score_distribution": {
                "mean": float(np.mean(scores)),
                "std":  float(np.std(scores)),
                "min":  float(np.min(scores)),
                "max":  float(np.max(scores)),
            },
        }
        self.results["calibration"] = out
        return out

    def get_all_results(self) -> Dict[str, Any]:
        return self.results


# ==================================================================
# Demo / Entry Point (Colab-safe — NO argparse)
# ==================================================================

TEST_QUERIES: List[str] = [
    "What is the relationship between diabetes and cardiovascular disease?",
    "List all treatments for Type 2 diabetes.",
    "What are the symptoms of myocardial infarction?",
    "How is hypertension diagnosed?",
    "What medications are used to treat asthma?",
]


def create_test_system() -> NeuroSymbolicSystem:
    """Create a fully mocked, dependency-light system for demos."""
    return NeuroSymbolicSystem(
        ontology_path=None,
        llm_model_name="mock",
        embedding_model_name="mock",
        threshold_config=ThresholdConfig.for_domain(DomainType.BIOMEDICAL),
        domain=DomainType.BIOMEDICAL,
        enable_async=False,
    )


def run_demo() -> None:
    """Run the full synchronous demo."""
    print("=" * 70)
    print("Neuro-Symbolic Framework Demonstration")
    print("=" * 70)

    system = create_test_system()

    print("\n" + "-" * 70)
    print("Processing Queries")
    print("-" * 70)

    for q in TEST_QUERIES:
        r = system.process_query(q, run_async=False)
        print(f"\nQuery: {q}")
        print(f"  Path:              {r.path.value}")
        print(f"  Fast Score:        {r.fast_score:.3f}")
        print(f"  Threshold (θ_low): {r.theta_low:.3f}")
        print(f"  Latency:           {r.latency_ms:.2f} ms")
        print(f"  Triples:           {len(r.triples)}")
        print(f"  Response:          {r.response[:100]}...")

    print("\n" + "-" * 70)
    print("System Statistics")
    print("-" * 70)
    stats = system.get_statistics()
    for k, v in stats.items():
        if isinstance(v, dict):
            print(f"\n  {k}:")
            for kk, vv in v.items():
                print(f"    {kk}: {vv:.4f}" if isinstance(vv, float)
                      else f"    {kk}: {vv}")
        else:
            print(f"  {k}: {v}")

    print("\n" + "-" * 70)
    print("Running Benchmark (num_runs=10 per query)")
    print("-" * 70)
    bench = BenchmarkSuite(system)
    lat = bench.run_latency_benchmark(TEST_QUERIES, num_runs=10)

    print("\nLatency Results")
    print(f"  Fast path  : {lat['fast_path']['mean_ms']:.2f} ms "
          f"± {lat['fast_path']['std_ms']:.2f} ms "
          f"(n={lat['fast_path']['count']})")
    print(f"  Corrections: {lat['correction_path']['mean_ms']:.2f} ms "
          f"± {lat['correction_path']['std_ms']:.2f} ms "
          f"(n={lat['correction_path']['count']})")
    print(f"  Throughput : {lat['throughput_qps']:.1f} queries/sec")

    print("\n" + "=" * 70)
    print("Demo Complete ✅")
    print("=" * 70)


async def run_async_demo() -> None:
    """Run the async demo (use `await run_async_demo()` in Colab)."""
    print("=" * 70)
    print("Async Mode Demonstration")
    print("=" * 70)

    system = create_test_system()
    system.enable_async = True

    for q in TEST_QUERIES:
        await system.process_query_async(q)

    await asyncio.sleep(0.1)

    stats = system.get_statistics()
    print(f"\nProcessed {stats['total_queries']} queries asynchronously.")
    print(f"  Fast path count : {stats['fast_path_count']}")
    print(f"  Slow path count : {stats['slow_path_count']}")
    print(f"  Corrections     : {stats['correction_count']}")

    items = await system.get_human_review_items()
    print(f"  Human review queue size: {len(items)}")

    print("\n" + "=" * 70)
    print("Async Demo Complete ✅")
    print("=" * 70)


# ------------------------------------------------------------------
# Auto-run the sync demo when this cell is executed in Colab.
# (No argparse, no CLI — works in Colab, Jupyter, and terminals.)
# ------------------------------------------------------------------
if __name__ == "__main__":
    run_demo()


Repository Not Found for url: https://huggingface.co/sentence-transformers/mock/resolve/main/modules.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.. Using mock.


Neuro-Symbolic Framework Demonstration

----------------------------------------------------------------------
Processing Queries
----------------------------------------------------------------------

Query: What is the relationship between diabetes and cardiovascular disease?
  Path:              correction
  Fast Score:        0.087
  Threshold (θ_low): 0.643
  Latency:           21.21 ms
  Triples:           2
  Response:          I need to verify some information. The following response may contain inconsistencies: 'Diabetes is ...

Query: List all treatments for Type 2 diabetes.
  Path:              correction
  Fast Score:        0.087
  Threshold (θ_low): 0.643
  Latency:           5.99 ms
  Triples:           2
  Response:          I need to verify some information. The following response may contain inconsistencies: 'Diabetes is ...

Query: What are the symptoms of myocardial infarction?
  Path:              correction
  Fast Score:        0.093
  Threshold (θ_low): 0.643
  L